# 06 · Deep-Cleaning Pipeline

Picks up from `integrated_conversations_ner_validated.csv` (output of notebook 05) and applies five additional cleaning stages:

| Stage | What it does |
|---|---|
| **A** | Template / boilerplate filtering — ChatDoctor sign-offs, MedDialog artefacts, LLM greeting loops |
| **B** | Near-duplicate detection with **MinHash LSH** (catches paraphrased templates that exact-dedup misses) |
| **C** | Entity normalisation — lowercase + spaCy lemmatise to collapse `pain` / `pains` / `Pain` variants |
| **D** | NER quality improvement — scispaCy `en_ner_bc5cdr_md` + `EntityLinker` (UMLS) for DISEASE/CHEMICAL; supplement with `en_core_sci_lg` for symptom/anatomy coverage |
| **E** | 200-turn stratified spot-check — reproducible random sample for manual review |

**Input:**  `data/processed/integrated_conversations_ner_validated.csv`  
**Outputs:**
- `data/processed/integrated_conversations_clean.csv` — fully cleaned dataset
- `data/processed/spotcheck_200.csv` — 200-turn sample for manual NER review
- `data/processed/entity_norm_map.csv` — mapping of raw → normalised entity strings

> **Note on scispaCy:** `en_ner_bc5cdr_md` is already installed in this environment (see notebook 04).  
> The UMLS EntityLinker requires `scispacy` ≥ 0.5 and an internet connection on first run to  
> download the UMLS KB (~1 GB). If offline, Stage D degrades gracefully to normalisation-only.

---
## 0 · Imports & Configuration

In [ ]:
import re
import json
import hashlib
import warnings
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ── Optional heavy deps — imported lazily in each stage ──────────────────────
# datasketch (MinHash), spacy / scispacy (NER), nltk (lemmatiser)

warnings.filterwarnings('ignore')
tqdm.pandas()

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR     = Path('../data/processed')
INPUT_CSV    = DATA_DIR / 'integrated_conversations_ner_validated.csv'
OUTPUT_CSV   = DATA_DIR / 'integrated_conversations_clean.csv'
SPOTCHECK    = DATA_DIR / 'spotcheck_200.csv'
NORM_MAP_CSV = DATA_DIR / 'entity_norm_map.csv'

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED  = 42

# ── MinHash settings ──────────────────────────────────────────────────────────
MINHASH_NUM_PERM    = 128   # permutations — higher = more accurate, slower
MINHASH_THRESHOLD   = 0.85  # Jaccard similarity threshold; 0.85 catches close paraphrases
MINHASH_SHINGLE_K   = 5     # character n-gram size for shingling

print('Paths configured:')
for p in [INPUT_CSV, OUTPUT_CSV, SPOTCHECK, NORM_MAP_CSV]:
    print(f'  {p}')

---
## 1 · Load Data

In [ ]:
df = pd.read_csv(
    INPUT_CSV,
    low_memory=False,
    dtype={'original_id': str, 'dialogue_id': str}
)

# Normalise key columns defensively
df['utterance']          = df['utterance'].fillna('').astype(str).str.strip()
df['extracted_entities'] = df['extracted_entities'].fillna('').astype(str)
df['entity_labels']      = df['entity_labels'].fillna('').astype(str)
df['source_dataset']     = df['source_dataset'].fillna('').astype(str)

PIPE_SEP = ' | '

def split_pipe(val):
    """Split a pipe-delimited entity string into a clean list."""
    if not val or val.strip() == '':
        return []
    return [t.strip() for t in val.split('|') if t.strip()]

df['ents_list']   = df['extracted_entities'].apply(split_pipe)
df['labels_list'] = df['entity_labels'].apply(split_pipe)

N_ORIG = len(df)
print(f'Loaded {N_ORIG:,} rows from {INPUT_CSV.name}')
print(f'Columns: {list(df.columns)}')
print(f'\nSource breakdown:')
print(df['source_dataset'].value_counts().to_string())

---
## Stage A · Template / Boilerplate Filtering

Three classes of boilerplate exist in this dataset:

1. **ChatDoctor sign-offs** — "Hi... thank you for consulting in chat doctor", "Hope I have answered your query", "chat doctor." suffix artefacts
2. **MedDialog greeting loops** — repeated identical greetings (`hello`, `hi`, `▌`)
3. **LLM templated openers** — "Hello! How can I assist you today?" (≥ 53 repetitions)

### Strategy
- **Whole-utterance drop:** Remove rows whose *entire content* is a known boilerplate string.
- **Inline scrubbing:** Strip boilerplate *suffixes/signatures* from longer utterances rather than deleting the whole turn.

In [ ]:
# ── A1. Compile pattern banks ─────────────────────────────────────────────────

# Phrases that, when they appear at the END of a turn, constitute a sign-off.
# These are stripped inline (the rest of the turn is kept).
SIGNOFF_PATTERNS = [
    # MedDialog / ChatDoctor sign-offs
    r'\bfurther clarifications and consultations on chat\s*doctor are welcome\.?\b',
    r'\bif you do not have any clarifications,? you can close the discussion and rate the answer\.?\b',
    r'\bi am happy to help any\s*time\.?\b',
    r'\bhope my answer was helpful for you\.?\b',
    r'\bhope i have answered your query\.?\b',
    r'\bhope this (?:will )?help(?:s)? you(?: in resolving your query)?\.?\b',
    r'\bthanks for your query\.?\b',
    r'\bthank you for (?:consulting|posting on) (?:in )?chat\s*doctor\.?\b',
    r'\bwish your(?:\s+\w+)? good health\.?\b',
    r'\bwish you (?:best|good)(?: of health)?\.?\b',
    r'\bchat\s*doctor\.\s*\(?\s*(?:mbbs|md|ms|mch|dnb)(?:\s*,\s*(?:mbbs|md|ms|mch|dnb))*\s*\)?',
    r'\bchat\s*doctor\.\s*$',
    # Generic MedDialog sign-offs
    r'\bkindly spare some time to rate my answer\b',
    r'\bplease do not hesitate to ask if you have any further questions?\.?\b',
    r'\bfeel free to (?:ask|consult)(?: me)?(?: again)? (?:anytime|if (?:you have|there are))\b',
    r'\btake care\.?\s*$',
    r'\bregards\s*[-–]\s*$',
    r'\bregards\.?\s*$',
    # "back to me for any follow-up" pattern
    r'\bback to me for any follow[\s-]?up queries anytime\.?\b',
]

# Compiled sign-off regex (IGNORECASE, multiline)
_SIGNOFF_RE = re.compile(
    '|'.join(f'({p})' for p in SIGNOFF_PATTERNS),
    flags=re.IGNORECASE | re.DOTALL
)

# ── A2. Whole-utterance drop list ─────────────────────────────────────────────
# Rows whose *entire* normalised utterance matches one of these are dropped
# (these carry zero information content).
WHOLE_DROP_EXACT = {
    # cursor / UI artefact
    '▌',
    # bare greetings (only when they are the *entire* utterance)
    'hi',
    'hello',
    'hello!',
    'hi!',
    'hi.',
    'hello.',
    # LLM canned opener
    'hello! how can i assist you today?',
    'hello! how can i help you today?',
    'hi! how can i assist you today?',
    'hi! how can i help you today?',
    # purely punctuation / whitespace
    '-',
    '--',
    '...',
    '.',
}

# Regex patterns for whole-utterance drops (applied after normalisation)
WHOLE_DROP_PATTERNS = [
    # Lines that are ONLY a chatdoctor ID badge (e.g. "chat doctor. (mbbs, ms)")
    r'^chat\s*doctor\.?\s*(?:\(?\s*(?:mbbs|md|ms|mch|dnb|dgo)[^)]*\)?)?\s*$',
    # Purely numeric or measurement-only turns
    r'^[\d\s.,;:%/\(\)\-\+]+$',
    # Single word that is a stopword
    r'^(?:yes|no|ok|okay|sure|thanks|thank\s*you|alright|alright\.|noted|noted\.)$',
]
_WHOLE_DROP_RE = re.compile(
    '|'.join(f'({p})' for p in WHOLE_DROP_PATTERNS),
    flags=re.IGNORECASE
)

print(f'Sign-off patterns loaded  : {len(SIGNOFF_PATTERNS)}')
print(f'Whole-drop exact strings  : {len(WHOLE_DROP_EXACT)}')
print(f'Whole-drop regex patterns : {len(WHOLE_DROP_PATTERNS)}')

In [ ]:
# ── A3. Apply cleaning ────────────────────────────────────────────────────────

def scrub_signoffs(text: str) -> str:
    """Strip boilerplate sign-off phrases from an utterance, keeping the rest."""
    cleaned = _SIGNOFF_RE.sub(' ', text)
    # Collapse runs of whitespace / trailing punctuation left by removal
    cleaned = re.sub(r'\s{2,}', ' ', cleaned).strip()
    cleaned = re.sub(r'[.\s]+$', '', cleaned).strip()
    return cleaned


def is_whole_drop(text: str) -> bool:
    """Return True if the entire utterance is boilerplate and should be dropped."""
    norm = text.strip().lower()
    if norm == '':
        return True
    if norm in WHOLE_DROP_EXACT:
        return True
    if _WHOLE_DROP_RE.fullmatch(norm):
        return True
    return False


# Step 1: inline sign-off scrubbing
df['utterance_clean'] = df['utterance'].progress_apply(scrub_signoffs)

# Step 2: flag rows whose *cleaned* content is now empty or whole-drop
df['_drop_template'] = df['utterance_clean'].apply(is_whole_drop)

n_template_drop = df['_drop_template'].sum()
n_scrubbed = (df['utterance_clean'] != df['utterance']).sum()

print(f'Utterances with sign-offs scrubbed : {n_scrubbed:,}')
print(f'Rows flagged for whole-drop        : {n_template_drop:,}')
print(f'  (of which null/empty originally) : {(df["utterance"] == "").sum():,}')

In [ ]:
# ── A4. Spot-check scrubbing results ─────────────────────────────────────────
# Show 8 before/after pairs to verify sign-off removal quality.

changed = df[df['utterance_clean'] != df['utterance']].sample(
    n=min(8, (df['utterance_clean'] != df['utterance']).sum()),
    random_state=RANDOM_SEED
)[['source_dataset', 'utterance', 'utterance_clean']]

pd.set_option('display.max_colwidth', 200)
for _, row in changed.iterrows():
    print(f"\n[{row['source_dataset']}]")
    print(f"  BEFORE: {row['utterance'][:300]}")
    print(f"  AFTER : {row['utterance_clean'][:300]}")

In [ ]:
# ── A5. Apply the drop and promote utterance_clean → utterance ────────────────
df_a = df[~df['_drop_template']].copy()
df_a['utterance'] = df_a['utterance_clean']
df_a = df_a.drop(columns=['utterance_clean', '_drop_template'])

print(f'Rows before Stage A : {len(df):,}')
print(f'Rows after  Stage A : {len(df_a):,}')
print(f'Dropped             : {len(df) - len(df_a):,} ({(len(df)-len(df_a))/len(df)*100:.2f}%)')

---
## Stage B · Near-Duplicate Detection with MinHash LSH

Exact deduplication (notebook 05) removed 533 rows.  
Paraphrased templates — e.g. slight wording variants of the same ChatDoctor canned response — will not be caught by exact matching.

### Method: MinHash + LSH
- Represent each utterance as a set of character **5-grams** (shingles).
- Compute a **MinHash** signature (128 permutations) for each row.
- Use **LSH Forest / banding** to find all pairs with estimated Jaccard similarity ≥ 0.85.
- Within each near-duplicate cluster, keep the **longest** utterance (most informative).

> **Install:** `pip install datasketch` if not already present.

In [ ]:
try:
    from datasketch import MinHash, MinHashLSH
    print('datasketch available')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'datasketch==1.6.5', '-q'])
    from datasketch import MinHash, MinHashLSH
    print('datasketch installed and imported')

In [ ]:
def make_shingles(text: str, k: int = MINHASH_SHINGLE_K) -> set:
    """Return set of character k-grams from a normalised text string."""
    t = text.lower().strip()
    # Remove punctuation runs to reduce noise
    t = re.sub(r'[^\w\s]', '', t)
    t = re.sub(r'\s+', ' ', t)
    return {t[i:i+k] for i in range(max(1, len(t) - k + 1))}


def make_minhash(shingles: set, num_perm: int = MINHASH_NUM_PERM) -> MinHash:
    m = MinHash(num_perm=num_perm)
    for s in shingles:
        m.update(s.encode('utf-8'))
    return m


print(f'Shingling and hashing {len(df_a):,} utterances …')
print(f'Settings: k={MINHASH_SHINGLE_K}, num_perm={MINHASH_NUM_PERM}, threshold={MINHASH_THRESHOLD}')

# ── Build MinHash index ───────────────────────────────────────────────────────
# We index only non-empty utterances with enough characters to shingle.
MIN_UTT_LEN = 20  # skip very short utterances from near-dup check

lsh = MinHashLSH(threshold=MINHASH_THRESHOLD, num_perm=MINHASH_NUM_PERM)

idx_to_minhash = {}
inserted_keys = set()

for i, (row_idx, row) in enumerate(tqdm(df_a.iterrows(), total=len(df_a), desc='MinHash')):
    utt = row['utterance']
    if len(utt) < MIN_UTT_LEN:
        continue
    shingles = make_shingles(utt)
    if not shingles:
        continue
    mh = make_minhash(shingles)
    key = str(row_idx)  # use the dataframe index as key
    idx_to_minhash[row_idx] = mh
    try:
        lsh.insert(key, mh)
        inserted_keys.add(row_idx)
    except ValueError:
        # duplicate key — shouldn't happen but guard anyway
        pass

print(f'\nInserted {len(inserted_keys):,} utterances into LSH index')

In [ ]:
# ── Query LSH to find near-duplicate clusters ─────────────────────────────────
#
# For each row already in the index, ask the LSH for its neighbours.
# Union-Find (disjoint sets) merges overlapping neighbour lists into clusters.

class UnionFind:
    def __init__(self): self.parent = {}
    def find(self, x):
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.parent[rb] = ra

uf = UnionFind()

for row_idx, mh in tqdm(idx_to_minhash.items(), desc='LSH query'):
    neighbours = lsh.query(mh)          # returns list of key-strings
    for n_key in neighbours:
        n_idx = int(n_key)
        if n_idx != row_idx:
            uf.union(row_idx, n_idx)

# Collect clusters (groups with > 1 member)
cluster_map = defaultdict(list)
for row_idx in inserted_keys:
    root = uf.find(row_idx)
    cluster_map[root].append(row_idx)

near_dup_clusters = {k: v for k, v in cluster_map.items() if len(v) > 1}
total_near_dup_rows = sum(len(v) for v in near_dup_clusters.values())

print(f'Near-duplicate clusters found : {len(near_dup_clusters):,}')
print(f'Rows involved in clusters     : {total_near_dup_rows:,}')

In [ ]:
# ── Inspect a sample of clusters before deciding what to drop ─────────────────
sample_clusters = list(near_dup_clusters.values())[:5]

for ci, cluster in enumerate(sample_clusters, 1):
    print(f'\n=== Cluster {ci} ({len(cluster)} members) ===')
    for row_idx in cluster[:4]:  # show max 4 per cluster
        utt = df_a.loc[row_idx, 'utterance']
        src = df_a.loc[row_idx, 'source_dataset']
        print(f"  [{src}] {utt[:200]}")

In [ ]:
# ── Drop near-duplicates: keep longest utterance per cluster ──────────────────
#
# Within each cluster, the longest utterance is the most informative.
# All shorter near-duplicates are removed.

rows_to_drop = set()

for root, cluster in near_dup_clusters.items():
    # Sort by utterance length descending; keep first, drop rest
    sorted_cluster = sorted(
        cluster,
        key=lambda i: len(df_a.loc[i, 'utterance']),
        reverse=True
    )
    rows_to_drop.update(sorted_cluster[1:])

df_b = df_a.drop(index=list(rows_to_drop)).copy()

print(f'Rows before Stage B : {len(df_a):,}')
print(f'Near-dup rows dropped : {len(rows_to_drop):,}')
print(f'Rows after  Stage B : {len(df_b):,}')
print(f'Drop rate           : {len(rows_to_drop)/len(df_a)*100:.2f}%')

---
## Stage C · Entity Normalisation (Lowercase + Lemmatise)

BC5CDR entities like `pain`, `Pain`, `pains`, `Pains` are all treated as separate strings downstream.  
This stage:
1. **Lowercases** all entity surface forms.
2. **Lemmatises** with spaCy's `en_core_web_sm` to collapse morphological variants (`pains → pain`, `infections → infection`, `steroids → steroid`).
3. Saves a `entity_norm_map.csv` mapping raw → normalised for auditability.
4. Updates `extracted_entities` in place with the normalised forms.

> Lemmatisation is applied to entity strings only (not full utterances), which is fast.

In [ ]:
import spacy

# Load a small model just for lemmatisation — no need for the biomedical model here
try:
    nlp_lemma = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
    print('en_core_web_sm loaded for lemmatisation')
except OSError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'
    ])
    nlp_lemma = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
    print('en_core_web_sm downloaded and loaded')

In [ ]:
def normalise_entity(raw: str) -> str:
    """
    Lowercase and lemmatise a single entity surface form.
    Multi-word entities: lemmatise each token, rejoin with a space.
    """
    lowered = raw.strip().lower()
    doc = nlp_lemma(lowered)
    lemmatised = ' '.join(token.lemma_ for token in doc)
    return lemmatised.strip()


# Collect unique raw entity strings across the whole dataset
all_raw_entities = set(
    ent
    for ents in df_b['ents_list']
    for ent in ents
    if ent.strip()
)

print(f'Unique raw entity strings: {len(all_raw_entities):,}')
print('Lemmatising …')

# Build the normalisation map (batch through spaCy for speed)
raw_list  = list(all_raw_entities)
norm_list = []

for doc in tqdm(nlp_lemma.pipe(raw_list, batch_size=512), total=len(raw_list), desc='Lemmatise'):
    norm_list.append(' '.join(t.lemma_ for t in doc).strip().lower())

norm_map = dict(zip(raw_list, norm_list))

# Save the map for audit / downstream use
norm_df = pd.DataFrame({
    'raw_entity':        raw_list,
    'normalised_entity': norm_list
})
norm_df['changed'] = norm_df['raw_entity'].str.lower().str.strip() != norm_df['normalised_entity']
norm_df.to_csv(NORM_MAP_CSV, index=False)

n_changed = norm_df['changed'].sum()
print(f'\nEntities that changed after normalisation: {n_changed:,} / {len(raw_list):,}')
print('\nSample changes:')
print(norm_df[norm_df['changed']].head(20).to_string(index=False))

In [ ]:
# ── Apply normalisation map to entity columns ─────────────────────────────────

def apply_norm(ents_list: list) -> list:
    return [norm_map.get(e, e.strip().lower()) for e in ents_list]


df_b['ents_norm'] = df_b['ents_list'].apply(apply_norm)

# Rebuild pipe-separated strings from normalised lists
df_b['extracted_entities'] = df_b['ents_norm'].apply(
    lambda lst: ' | '.join(lst) if lst else ''
)
# entity_labels stays aligned (same length lists)

# Verify alignment
misaligned = df_b.apply(
    lambda r: len(r['ents_norm']) != len(r['labels_list']), axis=1
).sum()
print(f'Misaligned rows after normalisation: {misaligned}  (should be 0)')

# Check collapse of variants
for raw_forms in [['pain', 'Pain', 'pains', 'Pains'],
                  ['infection', 'infections'],
                  ['steroid', 'steroids'],
                  ['vitamin d', 'Vitamin D', 'vitamin d3']]:
    print('\nVariant collapse:')
    for f in raw_forms:
        print(f'  {f!r:20s} → {norm_map.get(f, f.lower())!r}')

print(f'\nStage C complete — unique normalised entities: {len(set(norm_list)):,}')

---
## Stage D · NER Quality Improvement

### Problem summary
BC5CDR (`en_ner_bc5cdr_md`) knows only `DISEASE` and `CHEMICAL`. This causes:

| Entity | BC5CDR label | Correct label |
|---|---|---|
| `pain`, `throat`, `tooth` | DISEASE | SYMPTOM / ANATOMY |
| `smoking`, `alcohol` | CHEMICAL | BEHAVIOR |
| `tooth`, `liver`, `lung` | DISEASE | ANATOMY |

### Strategy (two complementary approaches)

**D1 — Rule-based label re-assignment**  
Curated vocabulary lists for SYMPTOM, ANATOMY, and BEHAVIOR.  
Fast, transparent, no extra model needed.

**D2 — scispaCy EntityLinker → UMLS CUI**  
For each entity string, query the UMLS knowledge base (via `scispacy`'s  
`EntityLinker` or a lookup against the UMLS MRCONSO table) to get a  
Concept Unique Identifier (CUI) and semantic type (TUI/STY).  
This enables downstream UMLS/MeSH grounding.

> **Note:** The UMLS linker downloads a ~1 GB cache on first run.  
> If offline, D2 is skipped and D1 still runs.

In [ ]:
# ── D1. Rule-based label re-assignment ───────────────────────────────────────
#
# Entities that BC5CDR mislabels.  We use normalised (lowercased) surface forms.
# Only entities currently labelled DISEASE or CHEMICAL are considered.

SYMPTOM_VOCAB = {
    # Pain / discomfort
    'pain', 'ache', 'aching', 'burning', 'stinging', 'tingling', 'numbness',
    'soreness', 'tenderness', 'cramp', 'spasm', 'pressure',
    # Respiratory
    'cough', 'sneeze', 'sneezing', 'wheeze', 'shortness of breath', 'sob',
    'breathlessness', 'dyspnea', 'sputum', 'phlegm', 'congestion', 'runny nose',
    # Gastrointestinal
    'nausea', 'vomiting', 'diarrhea', 'diarrhoea', 'constipation', 'bloating',
    'heartburn', 'indigestion', 'belching', 'flatulence',
    # General
    'fever', 'fatigue', 'tiredness', 'weakness', 'dizziness', 'vertigo',
    'headache', 'migraine', 'insomnia', 'sweating', 'chills', 'itching',
    'itchiness', 'rash', 'hives', 'swelling', 'edema', 'oedema',
    # Mood / cognitive
    'anxiety', 'depression', 'mood swing', 'irritability', 'confusion',
    'memory loss', 'brain fog', 'hallucination',
    # Other
    'bleeding', 'bruising', 'discharge', 'pus', 'jaundice', 'pallor', 'cyanosis',
    'sore',
}

ANATOMY_VOCAB = {
    # Head/neck
    'tooth', 'teeth', 'gum', 'tongue', 'throat', 'tonsil', 'tonsils',
    'neck', 'ear', 'ears', 'nose', 'eye', 'eyes', 'scalp', 'jaw',
    # Chest/abdomen
    'chest', 'breast', 'nipple', 'abdomen', 'stomach', 'belly', 'navel',
    'groin', 'pelvis',
    # Back/limbs
    'back', 'spine', 'shoulder', 'arm', 'elbow', 'wrist', 'hand', 'finger',
    'thumb', 'hip', 'knee', 'ankle', 'foot', 'toe', 'heel', 'calf', 'thigh',
    'leg', 'buttock',
    # Organs / systems
    'liver', 'kidney', 'lung', 'lungs', 'heart', 'brain', 'spleen',
    'pancreas', 'gallbladder', 'bladder', 'uterus', 'ovary', 'ovaries',
    'prostate', 'thyroid', 'adrenal', 'colon', 'rectum', 'appendix',
    'trachea', 'larynx', 'esophagus', 'oesophagus', 'duodenum', 'ileum',
    'jejunum', 'cecum', 'sigmoid', 'anus', 'vagina', 'cervix', 'penis',
    'testis', 'testes', 'bone', 'cartilage', 'tendon', 'ligament', 'muscle',
    'nerve', 'artery', 'vein', 'capillary', 'lymph node', 'lymph',
    # Skin
    'skin', 'nail', 'nails', 'hair', 'scalp',
}

BEHAVIOR_VOCAB = {
    'smoking', 'smoke', 'tobacco', 'cigarette', 'vaping', 'chewing tobacco',
    'alcohol', 'drinking', 'alcoholism', 'drug use', 'substance use',
    'recreational drug', 'heroin', 'cocaine', 'methamphetamine', 'cannabis',
    'marijuana', 'sedentary', 'exercise', 'physical activity',
}

# Map normalised entity → override label
LABEL_OVERRIDE = {}
for e in SYMPTOM_VOCAB:  LABEL_OVERRIDE[e] = 'SYMPTOM'
for e in ANATOMY_VOCAB:  LABEL_OVERRIDE[e] = 'ANATOMY'
for e in BEHAVIOR_VOCAB: LABEL_OVERRIDE[e] = 'BEHAVIOR'

print(f'Label override vocabulary:')
print(f'  SYMPTOM  : {len(SYMPTOM_VOCAB):>4} terms')
print(f'  ANATOMY  : {len(ANATOMY_VOCAB):>4} terms')
print(f'  BEHAVIOR : {len(BEHAVIOR_VOCAB):>4} terms')

In [ ]:
def relabel_entities(ents_norm: list, labels: list) -> list:
    """
    Apply rule-based label overrides to a (entity, label) pair list.
    Only DISEASE/CHEMICAL entities are candidates for relabelling.
    Returns the updated labels list.
    """
    new_labels = []
    for ent, lbl in zip(ents_norm, labels):
        override = LABEL_OVERRIDE.get(ent.strip().lower())
        if override and lbl in ('DISEASE', 'CHEMICAL'):
            new_labels.append(override)
        else:
            new_labels.append(lbl)
    return new_labels


df_b['labels_updated'] = df_b.apply(
    lambda r: relabel_entities(r['ents_norm'], r['labels_list']),
    axis=1
)

# Count how many entities were relabelled
n_relabelled = sum(
    sum(a != b for a, b in zip(old, new))
    for old, new in zip(df_b['labels_list'], df_b['labels_updated'])
)

# Show the label distribution before and after
all_old_labels = [l for lbls in df_b['labels_list']    for l in lbls]
all_new_labels = [l for lbls in df_b['labels_updated'] for l in lbls]

print(f'\nEntities relabelled by rule-based override: {n_relabelled:,}')
print('\nLabel distribution BEFORE relabelling:')
print(pd.Series(all_old_labels).value_counts().to_string())
print('\nLabel distribution AFTER relabelling:')
print(pd.Series(all_new_labels).value_counts().to_string())

# Update the column
df_b['entity_labels'] = df_b['labels_updated'].apply(
    lambda lst: ' | '.join(lst) if lst else ''
)

In [ ]:
# ── D2. scispaCy EntityLinker — UMLS CUI lookup ───────────────────────────────
#
# Attempts to add a `umls_cuis` column: pipe-separated UMLS CUIs per entity.
# Gracefully skips if scispacy or the UMLS linker is unavailable.
#
# The EntityLinker works on full sentences; here we run it on the *entity*
# surface form only (fast, avoids re-processing full utterances).
# For production, run it on full utterances and use the linker scores.

UMLS_AVAILABLE = False

try:
    import scispacy  # noqa: F401
    from scispacy.linking import EntityLinker  # noqa: F401

    # Load the biomedical model (already installed from notebook 04)
    nlp_sci = spacy.load('en_ner_bc5cdr_md')

    # Add the UMLS entity linker
    # resolve_abbreviations=True also resolves acronyms in context
    nlp_sci.add_pipe(
        'scispacy_linker',
        config={
            'resolve_abbreviations': True,
            'linker_name': 'umls',
            'threshold': 0.80,   # min similarity score to accept a CUI
            'filter_for_definitions': False,
        }
    )
    linker = nlp_sci.get_pipe('scispacy_linker')
    UMLS_AVAILABLE = True
    print('scispaCy UMLS EntityLinker loaded successfully')

except Exception as e:
    print(f'UMLS EntityLinker NOT available ({type(e).__name__}: {e})')
    print('Stage D2 will be skipped — D1 (rule-based) results are retained.')
    print()
    print('To enable UMLS linking, run:')
    print('  pip install scispacy==0.5.5')
    print('  pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.5/en_ner_bc5cdr_md-0.5.5.tar.gz')
    print('  # UMLS KB will download automatically on first pipe call (~1 GB)')

In [ ]:
if UMLS_AVAILABLE:
    # ── Run UMLS linking on unique normalised entities (not full utterances) ──
    # This is a fast approximation; for full-context linking, process utterances.

    unique_ents = list({
        e for ents in df_b['ents_norm'] for e in ents if e.strip()
    })
    print(f'Running UMLS linker on {len(unique_ents):,} unique entity strings …')

    cui_map = {}  # entity_string -> top CUI (or '' if not found)

    for doc in tqdm(
        nlp_sci.pipe(unique_ents, batch_size=256, disable=['ner']),
        total=len(unique_ents),
        desc='UMLS link'
    ):
        # The linker attaches kb_ents to tokens via the entity spans
        # When passed a short string, it may or may not produce spans
        all_cuis = []
        for ent in doc.ents:
            for kb_ent in ent._.kb_ents:
                all_cuis.append(kb_ent[0])  # (CUI, score)
        cui_map[doc.text] = all_cuis[0] if all_cuis else ''

    # Add CUI column
    df_b['umls_cuis'] = df_b['ents_norm'].apply(
        lambda ents: ' | '.join(cui_map.get(e, '') for e in ents) if ents else ''
    )

    n_linked = sum(1 for v in cui_map.values() if v)
    print(f'\nEntities with a UMLS CUI : {n_linked:,} / {len(unique_ents):,} ({n_linked/len(unique_ents)*100:.1f}%)')
    print('\nSample CUI mappings:')
    sample_linked = [(k, v) for k, v in cui_map.items() if v][:15]
    for ent, cui in sample_linked:
        kb_name = linker.kb.cui_to_entity.get(cui, None)
        name_str = kb_name.canonical_name if kb_name else '?'
        print(f'  {ent:30s} → {cui}  ({name_str})')
else:
    df_b['umls_cuis'] = ''
    print('umls_cuis column set to empty (linker not available)')

### D3 · Why `en_core_sci_lg` instead of or alongside BC5CDR?

`en_core_sci_lg` is a general biomedical NER model trained on the JNLPBA/BioNLP corpora.  
It labels **18 entity types** including `CELL_TYPE`, `PROTEIN`, `GENE`, `DNA`, `RNA`,  
`CELL_LINE`, and broader `DISEASE` / `CHEMICAL` spans.  
It does **not** natively add `SYMPTOM` or `ANATOMY` labels either, but its DISEASE spans  
are broader and cover symptom-like terms more accurately than BC5CDR.

**Recommended workflow for symptom/anatomy coverage:**
1. Run **rule-based relabelling** (D1 above) — immediate, transparent.
2. Run **`en_ner_bc5cdr_md` + UMLS linker** (D2) — CUI semantic type (TUI) classifies entities precisely.
3. Optionally run **`en_core_sci_lg`** in an ensemble — union of spans from both models.

The code below shows how to install and run `en_core_sci_lg` for comparison.

In [ ]:
# ── D3. en_core_sci_lg ensemble (optional, comment out if not needed) ─────────
#
# Install:
#   pip install scispacy==0.5.5
#   pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.5/en_core_sci_lg-0.5.5.tar.gz
#
# This cell loads the model and runs it on a 500-row sample for comparison.

SCI_LG_AVAILABLE = False

try:
    nlp_sci_lg = spacy.load('en_core_sci_lg')
    SCI_LG_AVAILABLE = True
    print('en_core_sci_lg loaded')
except OSError:
    print('en_core_sci_lg not installed — skipping ensemble comparison.')
    print('Install with:')
    print('  pip install scispacy==0.5.5')
    print('  pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/'
          'releases/v0.5.5/en_core_sci_lg-0.5.5.tar.gz')

if SCI_LG_AVAILABLE:
    sample_utts = df_b['utterance'].dropna().sample(500, random_state=RANDOM_SEED).tolist()
    sci_lg_results = []

    for doc in tqdm(nlp_sci_lg.pipe(sample_utts, batch_size=64), total=500, desc='sci_lg'):
        sci_lg_results.append([(ent.text, ent.label_) for ent in doc.ents])

    # Count label distribution from sci_lg
    sci_lg_labels = Counter(
        lbl
        for row in sci_lg_results
        for _, lbl in row
    )
    print('\nen_core_sci_lg label distribution on 500-row sample:')
    for lbl, cnt in sci_lg_labels.most_common():
        print(f'  {lbl:20s} {cnt:>6,}')
    print('\nNote: sci_lg DISEASE spans often include symptoms BC5CDR misses.')

---
## Stage E · 200-Turn Stratified Spot-Check

Generates a reproducible 200-turn sample stratified by:
- Source dataset (proportional)
- Speaker (user / assistant balanced)
- Has-entity flag (mix of entity-bearing and entity-free turns)

The output CSV includes extra columns (`_review_label`, `_review_notes`) for manual annotation.

In [ ]:
# ── E1. Stratification setup ──────────────────────────────────────────────────

N_SAMPLE = 200

df_b['has_entity'] = df_b['ents_norm'].apply(lambda x: len(x) > 0)

# Stratum = (source_dataset, speaker, has_entity)
df_b['_stratum'] = (
    df_b['source_dataset'] + '::' +
    df_b['speaker'] + '::' +
    df_b['has_entity'].map({True: 'has_ent', False: 'no_ent'})
)

strata_counts = df_b['_stratum'].value_counts()
print(f'Strata defined: {len(strata_counts)}')
print(strata_counts.to_string())

In [ ]:
# ── E2. Proportional allocation ───────────────────────────────────────────────

stratum_weights = strata_counts / strata_counts.sum()
stratum_alloc   = (stratum_weights * N_SAMPLE).round().astype(int)

# Fix rounding to get exactly N_SAMPLE
diff = N_SAMPLE - stratum_alloc.sum()
if diff != 0:
    # Add/remove from the largest stratum
    top = stratum_alloc.idxmax()
    stratum_alloc[top] += diff

print(f'\nAllocated {stratum_alloc.sum()} rows across strata:')
print(stratum_alloc.to_string())

In [ ]:
# ── E3. Draw the sample ───────────────────────────────────────────────────────

rng = np.random.default_rng(RANDOM_SEED)
sample_parts = []

for stratum, n_draw in stratum_alloc.items():
    pool = df_b[df_b['_stratum'] == stratum]
    actual = min(n_draw, len(pool))
    if actual > 0:
        drawn = pool.sample(n=actual, random_state=RANDOM_SEED)
        sample_parts.append(drawn)

spotcheck_df = pd.concat(sample_parts).sample(frac=1, random_state=RANDOM_SEED)

# Add annotation columns for manual review
spotcheck_df = spotcheck_df.copy()
spotcheck_df['_review_label'] = ''    # reviewer fills: CORRECT / WRONG / PARTIAL
spotcheck_df['_review_notes'] = ''    # reviewer fills: free text

# Select the columns useful for review
review_cols = [
    'dialogue_id', 'turn_id', 'speaker', 'source_dataset',
    'utterance', 'extracted_entities', 'entity_labels', 'umls_cuis',
    '_review_label', '_review_notes'
]
review_cols = [c for c in review_cols if c in spotcheck_df.columns]

spotcheck_df[review_cols].to_csv(SPOTCHECK, index=False)

print(f'Spot-check sample saved: {SPOTCHECK}')
print(f'Total rows              : {len(spotcheck_df)}')
print(f'\nSource breakdown:')
print(spotcheck_df['source_dataset'].value_counts().to_string())
print(f'\nSpeaker breakdown:')
print(spotcheck_df['speaker'].value_counts().to_string())
print(f'\nHas-entity breakdown:')
print(spotcheck_df['has_entity'].value_counts().to_string())

In [ ]:
# ── E4. Print 10 sample rows for a quick visual review ───────────────────────

pd.set_option('display.max_colwidth', 120)
print('=== 10 randomly sampled spot-check turns ===')
for _, row in spotcheck_df.sample(10, random_state=RANDOM_SEED).iterrows():
    print(f"\n[{row['source_dataset']} | turn {row['turn_id']} | {row['speaker']}]")
    print(f"  UTT  : {row['utterance'][:200]}")
    print(f"  ENTS : {row['extracted_entities'][:120]}")
    print(f"  LBLS : {row['entity_labels'][:120]}")
    if 'umls_cuis' in row:
        print(f"  UMLS : {str(row['umls_cuis'])[:120]}")

---
## Final Save

In [ ]:
# ── Finalise column set and save ──────────────────────────────────────────────

# Recompute entity_count from normalised/relabelled lists
df_b['entity_count'] = df_b['ents_norm'].apply(len)

# Drop internal working columns
drop_cols = ['ents_list', 'labels_list', 'ents_norm', 'labels_updated',
             'has_entity', '_stratum', 'utt_norm']
df_final = df_b.drop(columns=[c for c in drop_cols if c in df_b.columns])

# Ensure entity_labels is up to date
# (already updated in Stage D, but confirm)
assert 'entity_labels' in df_final.columns

df_final.to_csv(OUTPUT_CSV, index=False)

print('=== Pipeline Complete ===')
print(f'Input rows              : {N_ORIG:,}')
print(f'After Stage A (template): {len(df_a):,}  (−{N_ORIG - len(df_a):,})')
print(f'After Stage B (near-dup): {len(df_b):,}  (−{len(df_a) - len(df_b):,})')
print(f'After Stage C/D (norm)  : {len(df_final):,}  (same, in-place)')
print(f'\nSaved to:')
print(f'  {OUTPUT_CSV}')
print(f'  {SPOTCHECK}')
print(f'  {NORM_MAP_CSV}')
print(f'\nFinal unique normalised entities: '
      f"{df_final['extracted_entities'].apply(lambda x: [t.strip() for t in x.split('|') if t.strip()]).explode().nunique():,}")

---
## Summary Report

In [ ]:
from IPython.display import display, HTML

label_counts_final = Counter(
    lbl
    for row in df_final['entity_labels']
    for lbl in (row.split(' | ') if row else [])
    if lbl.strip()
)

total_ents = sum(label_counts_final.values())

report = pd.DataFrame([
    {'Metric': 'Total rows (turns)',                  'Value': f'{len(df_final):,}'},
    {'Metric': 'Rows dropped — template/boilerplate', 'Value': f'{N_ORIG - len(df_a):,}'},
    {'Metric': 'Rows dropped — near-duplicate (MinHash LSH)', 'Value': f'{len(rows_to_drop):,}'},
    {'Metric': 'Total entity mentions (final)',       'Value': f'{total_ents:,}'},
    {'Metric': 'Entity mentions — DISEASE',           'Value': f"{label_counts_final.get('DISEASE', 0):,}"},
    {'Metric': 'Entity mentions — CHEMICAL',          'Value': f"{label_counts_final.get('CHEMICAL', 0):,}"},
    {'Metric': 'Entity mentions — SYMPTOM (new)',     'Value': f"{label_counts_final.get('SYMPTOM', 0):,}"},
    {'Metric': 'Entity mentions — ANATOMY (new)',     'Value': f"{label_counts_final.get('ANATOMY', 0):,}"},
    {'Metric': 'Entity mentions — BEHAVIOR (new)',    'Value': f"{label_counts_final.get('BEHAVIOR', 0):,}"},
    {'Metric': 'Unique normalised entity strings',    'Value': f'{len(set(norm_list)):,}'},
    {'Metric': 'Spot-check sample size',              'Value': '200 turns'},
    {'Metric': 'UMLS CUI linking',                   'Value': 'Enabled' if UMLS_AVAILABLE else 'Skipped (linker not installed)'},
])

display(HTML(report.to_html(index=False)))